# Tutorial 1: Auditing `disba` Fundamental-Mode Root Skipping on Shallow LVZ Profiles

This pure-CPU tutorial reproduces the core forward-solver pitfall reported in Section 3 of the manuscript:
1. Load representative shallow low-velocity-zone (LVZ) profile `#9391` (`CurveVelA`) from `results_summary/root_skipping_diagnostics.json`.
2. Evaluate fundamental-mode Rayleigh phase velocity with default `dc = 5.0e-3 km/s` vs. converged `dc = 2.5e-5 km/s`.
3. Confirm that refining `dc` alone collapses apparent held-out misfit from **63.84% to 0.65%** without modifying a single velocity layer.
4. Sweep all 6 archived shallow LVZ examples to verify complete Mode-0 recovery.

In [ ]:
import json
from pathlib import Path
import numpy as np

from swi_tta.evaluation.metrics import disba_misfit_ho
from swi_tta.physics.disba_referee import (
    DEFAULT_DEEP_DC_KM_S,
    DEFAULT_SHALLOW_DC_KM_S,
    forward_dispersion_diag,
    vs_to_vp_rho,
)

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'tutorials' else Path.cwd()
rs = json.loads((REPO_ROOT / 'results_summary' / 'root_skipping_diagnostics.json').read_text(encoding='utf-8'))
depth_km = np.asarray(rs['typical_lvz_depth_km'], dtype=np.float64)
ex = rs['typical_lvz_examples'][0]
print(f"Loaded sample #{ex['sample_idx']} ({ex['structure_type']}), layers={depth_km.size}, z_res={ex['z_res_km']:.3f} km")

In [ ]:
vs_profile = np.asarray(ex['profiles']['vs_refined'], dtype=np.float64)
periods_s = np.asarray(ex['periods_s'], dtype=np.float64)
phase_obs = np.asarray(ex['phase_obs_km_s'], dtype=np.float64)
is_ho = np.asarray(ex['is_heldout'], dtype=bool)

# Inspect Brocher (2005) empirical Vp and density mapping
vp, rho = vs_to_vp_rho(vs_profile, depth_km)
print(f'Vs range: [{vs_profile.min():.3f}, {vs_profile.max():.3f}] km/s | Vp range: [{vp.min():.3f}, {vp.max():.3f}] km/s')

# Compare coarse default dc=5e-3 vs converged shallow dc=2.5e-5
res_coarse = forward_dispersion_diag(vs_profile, depth_km, periods_s, mode=0, compute_group=False, dc=DEFAULT_DEEP_DC_KM_S)
res_conv = forward_dispersion_diag(vs_profile, depth_km, periods_s, mode=0, compute_group=False, dc=DEFAULT_SHALLOW_DC_KM_S)

obs_mask = np.ones_like(phase_obs, dtype=np.float64)
ho_mask = is_ho.astype(np.float64)
ho_coarse = disba_misfit_ho(res_coarse['phase_vel'], phase_obs, obs_mask, ho_mask)
ho_conv = disba_misfit_ho(res_conv['phase_vel'], phase_obs, obs_mask, ho_mask)

print(f'Default   dc={DEFAULT_DEEP_DC_KM_S:g} km/s -> HO misfit = {ho_coarse*100:.2f}% (Mode-1 jump)')
print(f'Converged dc={DEFAULT_SHALLOW_DC_KM_S:g} km/s -> HO misfit = {ho_conv*100:.2f}% (Mode-0 recovered)')

In [ ]:
# Verify all 6 representative shallow LVZ profiles
for item in rs['typical_lvz_examples']:
    vs_p = np.asarray(item['profiles']['vs_refined'], dtype=np.float64)
    per = np.asarray(item['periods_s'], dtype=np.float64)
    obs = np.asarray(item['phase_obs_km_s'], dtype=np.float64)
    ho = np.asarray(item['is_heldout'], dtype=np.float64)
    om = np.ones_like(obs, dtype=np.float64)
    m_old = disba_misfit_ho(forward_dispersion_diag(vs_p, depth_km, per, dc=5e-3)['phase_vel'], obs, om, ho)
    m_new = disba_misfit_ho(forward_dispersion_diag(vs_p, depth_km, per, dc=2.5e-5)['phase_vel'], obs, om, ho)
    print(f"Sample #{item['sample_idx']:>4d} ({item['structure_type']:<10s}): {m_old*100:6.2f}% -> {m_new*100:4.2f}%")